In [51]:
# ============================================================
# STEP 1 - IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print("Libraries imported successfully.")

Libraries imported successfully.


In [52]:
# ============================================================
# STEP 2 - LOAD ENGINEERED FEATURES
# ============================================================

FEATURE_PATH = "../data/processed/student_features.csv"

student_features = pd.read_csv(FEATURE_PATH)

print("Engineered features loaded successfully.")
print("Dataset shape:", student_features.shape)

Engineered features loaded successfully.
Dataset shape: (15, 29)


In [53]:
# ============================================================
# STEP 3 - INSPECT ENGINEERED DATA
# ============================================================

print("Columns:")
print(student_features.columns.tolist())

print("\nFirst 5 rows:")
display(student_features.head())

Columns:
['Student_ID', 'Average_Score', 'Average_Attempts', 'Average_Time_Taken', 'Total_Activities', 'Completion_Rate', 'Pre_Test', 'Post_Test', 'Unit_Test', 'Monthly_Test', 'Term_Result', 'Motor_Skills_Score', 'Creativity_Score', 'Interactive_Engagement', 'Test_Improvement', 'Participation', 'Attention_Level', 'Confidence', 'Collaboration', 'Learning_Difficulty', 'Learning_Style', 'Participation_Score', 'Attention_Score', 'Confidence_Score', 'Collaboration_Score', 'Historical_Lessons', 'Average_Sequence_No', 'Completed_Historical_Lessons', 'Historical_Completion_Rate']

First 5 rows:


,Student_ID,Average_Score,Average_Attempts,Average_Time_Taken,Total_Activities,Completion_Rate,Pre_Test,Post_Test,Unit_Test,Monthly_Test,...,Learning_Difficulty,Learning_Style,Participation_Score,Attention_Score,Confidence_Score,Collaboration_Score,Historical_Lessons,Average_Sequence_No,Completed_Historical_Lessons,Historical_Completion_Rate
0,ST001,78.00,1.75,11.25,4,100.0,50.0,77.5,74.0,73.5,...,Reading difficulty,Visual learner,3,2,3,3,4,2.5,2,50.0
1,ST002,78.75,1.50,11.25,4,100.0,47.5,65.0,65.0,68.0,...,Slow writing,Practical learner,1,1,2,2,4,2.5,2,50.0
2,ST003,92.25,1.00,16.00,4,100.0,60.0,82.5,80.0,81.0,...,None identified,Visual learner,3,3,3,3,4,2.5,2,50.0
3,ST004,68.75,2.00,11.75,4,75.0,37.5,57.5,58.0,61.0,...,Number recognition,Practical learner,2,2,2,3,4,2.5,1,25.0
4,ST005,95.25,1.00,16.00,4,100.0,60.0,85.0,85.0,88.5,...,None identified,Creative learner,3,3,3,4,4,2.5,3,75.0


In [54]:
# ============================================================
# STEP 4 - FINAL DATA QUALITY CHECK
# ============================================================

print("Missing values:",
      student_features.isnull().sum().sum())

print("Duplicate rows:",
      student_features.duplicated().sum())

print("Dataset shape:",
      student_features.shape)

Missing values: 0
Duplicate rows: 0
Dataset shape: (15, 29)


In [55]:
# ============================================================
# STEP 4.5 - LOAD HISTORICAL LEARNING PATH DATA
# ============================================================

import pandas as pd
import os

PROCESSED_PATH = "../data/processed"

historical_clean = pd.read_csv(
    f"{PROCESSED_PATH}/historical_clean.csv"
)

print("Historical learning-path data loaded successfully.")
print("Shape:", historical_clean.shape)

display(historical_clean.head())

Historical learning-path data loaded successfully.
Shape: (60, 6)


,Student_ID,Lesson_ID,Sequence_No,Status,Activity_Sequence,Outcome
0,ST001,LES001,1,Completed,Counting → Numbers,Successful
1,ST001,LES003,2,Completed,Letters → Sounds → Words,Needs Practice
2,ST001,LES004,3,Not Started,Reading Sentences,Waiting
3,ST001,LES005,4,In Progress,Number Matching → Number Puzzle,Improving
4,ST002,LES001,1,Completed,Counting Objects → Number Recognition,Successful


In [56]:
# ============================================================
# STEP 5 - INSPECT LEARNING OUTCOME TARGET
# ============================================================

print("Learning outcome distribution:")
print(historical_clean["Outcome"].value_counts())

print("\nUnique learning outcomes:")
print(historical_clean["Outcome"].unique())

Learning outcome distribution:
Outcome
Successful        21
Improving         16
Waiting           13
Needs Practice    10
Name: count, dtype: int64

Unique learning outcomes:
['Successful' 'Needs Practice' 'Waiting' 'Improving']


In [57]:
# ============================================================
# STEP 6 - CREATE STUDENT-LEVEL TARGET
# ============================================================

# Count each outcome for every student
outcome_counts = pd.crosstab(
    historical_clean["Student_ID"],
    historical_clean["Outcome"]
)

print("Outcome counts per student:")
display(outcome_counts)

Outcome counts per student:


Outcome,Improving,Needs Practice,Successful,Waiting
Student_ID,,,,
ST001,1,1,1,1
ST002,1,0,2,1
ST003,1,0,2,1
ST004,1,1,0,2
ST005,1,0,3,0
ST006,1,2,0,1
ST007,1,1,1,1
ST008,1,0,3,0
ST009,1,1,1,1


In [58]:
# ============================================================
# STEP 6.1 - SELECT DOMINANT OUTCOME FOR EACH STUDENT
# ============================================================

student_target = (
    historical_clean
    .groupby("Student_ID")["Outcome"]
    .agg(lambda x: x.value_counts().idxmax())
    .reset_index()
)

student_target.columns = ["Student_ID", "Target_Outcome"]

print("Student-level target:")
display(student_target)

Student-level target:


,Student_ID,Target_Outcome
0,ST001,Successful
1,ST002,Successful
2,ST003,Successful
3,ST004,Waiting
4,ST005,Successful
5,ST006,Needs Practice
6,ST007,Needs Practice
7,ST008,Successful
8,ST009,Needs Practice
9,ST010,Successful


In [59]:
# ============================================================
# STEP 6.2 - CHECK TARGET DISTRIBUTION
# ============================================================

print("Student-level target distribution:")
print(student_target["Target_Outcome"].value_counts())

print("\nTarget distribution (%):")
print(
    student_target["Target_Outcome"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Student-level target distribution:
Target_Outcome
Successful        9
Needs Practice    5
Waiting           1
Name: count, dtype: int64

Target distribution (%):
Target_Outcome
Successful        60.00
Needs Practice    33.33
Waiting            6.67
Name: proportion, dtype: float64


In [60]:
# ============================================================
# STEP 6.3 - VERIFY STUDENT-LEVEL TARGET
# ============================================================

print("Number of students:", len(student_target))
print("Unique students:", student_target["Student_ID"].nunique())
print("Number of target classes:", student_target["Target_Outcome"].nunique())

print("\nTarget classes:")
print(student_target["Target_Outcome"].unique())

Number of students: 15
Unique students: 15
Number of target classes: 3

Target classes:
['Successful' 'Waiting' 'Needs Practice']


In [61]:
# ============================================================
# STEP 7 - MERGE FEATURES WITH TARGET
# ============================================================

model_data = student_features.merge(
    student_target,
    on="Student_ID",
    how="inner"
)

print("Model dataset created successfully.")
print("Shape:", model_data.shape)

display(model_data.head())

Model dataset created successfully.
Shape: (15, 30)


,Student_ID,Average_Score,Average_Attempts,Average_Time_Taken,Total_Activities,Completion_Rate,Pre_Test,Post_Test,Unit_Test,Monthly_Test,...,Learning_Style,Participation_Score,Attention_Score,Confidence_Score,Collaboration_Score,Historical_Lessons,Average_Sequence_No,Completed_Historical_Lessons,Historical_Completion_Rate,Target_Outcome
0,ST001,78.00,1.75,11.25,4,100.0,50.0,77.5,74.0,73.5,...,Visual learner,3,2,3,3,4,2.5,2,50.0,Successful
1,ST002,78.75,1.50,11.25,4,100.0,47.5,65.0,65.0,68.0,...,Practical learner,1,1,2,2,4,2.5,2,50.0,Successful
2,ST003,92.25,1.00,16.00,4,100.0,60.0,82.5,80.0,81.0,...,Visual learner,3,3,3,3,4,2.5,2,50.0,Successful
3,ST004,68.75,2.00,11.75,4,75.0,37.5,57.5,58.0,61.0,...,Practical learner,2,2,2,3,4,2.5,1,25.0,Waiting
4,ST005,95.25,1.00,16.00,4,100.0,60.0,85.0,85.0,88.5,...,Creative learner,3,3,3,4,4,2.5,3,75.0,Successful


In [62]:
# ============================================================
# STEP 7.1 - VERIFY MERGED DATA
# ============================================================

print("Model data shape:")
print(model_data.shape)

print("\nTotal missing values:")
print(model_data.isnull().sum().sum())

print("\nDuplicate rows:")
print(model_data.duplicated().sum())

print("\nTarget distribution:")
print(model_data["Target_Outcome"].value_counts())

Model data shape:
(15, 30)

Total missing values:
0

Duplicate rows:
0

Target distribution:
Target_Outcome
Successful        9
Needs Practice    5
Waiting           1
Name: count, dtype: int64


In [63]:
# ============================================================
# STEP 8 - SEPARATE FEATURES AND TARGET
# ============================================================

# Target variable
y = model_data["Target_Outcome"]

# Remove ID and target from input features
X = model_data.drop(
    columns=["Student_ID", "Target_Outcome"]
)

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeature columns:")
print(X.columns.tolist())

print("\nTarget classes:")
print(y.unique())

Features shape: (15, 28)
Target shape: (15,)

Feature columns:
['Average_Score', 'Average_Attempts', 'Average_Time_Taken', 'Total_Activities', 'Completion_Rate', 'Pre_Test', 'Post_Test', 'Unit_Test', 'Monthly_Test', 'Term_Result', 'Motor_Skills_Score', 'Creativity_Score', 'Interactive_Engagement', 'Test_Improvement', 'Participation', 'Attention_Level', 'Confidence', 'Collaboration', 'Learning_Difficulty', 'Learning_Style', 'Participation_Score', 'Attention_Score', 'Confidence_Score', 'Collaboration_Score', 'Historical_Lessons', 'Average_Sequence_No', 'Completed_Historical_Lessons', 'Historical_Completion_Rate']

Target classes:
['Successful' 'Waiting' 'Needs Practice']


In [64]:
# ============================================================
# STEP 8.1 - IDENTIFY NUMERICAL AND CATEGORICAL FEATURES
# ============================================================

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nNumber of numerical features:")
print(len(numerical_features))

print("\nCategorical features:")
print(categorical_features)

print("\nNumber of categorical features:")
print(len(categorical_features))

Numerical features:
['Average_Score', 'Average_Attempts', 'Average_Time_Taken', 'Total_Activities', 'Completion_Rate', 'Pre_Test', 'Post_Test', 'Unit_Test', 'Monthly_Test', 'Term_Result', 'Motor_Skills_Score', 'Creativity_Score', 'Interactive_Engagement', 'Test_Improvement', 'Participation_Score', 'Attention_Score', 'Confidence_Score', 'Collaboration_Score', 'Historical_Lessons', 'Average_Sequence_No', 'Completed_Historical_Lessons', 'Historical_Completion_Rate']

Number of numerical features:
22

Categorical features:
['Participation', 'Attention_Level', 'Confidence', 'Collaboration', 'Learning_Difficulty', 'Learning_Style']

Number of categorical features:
6


In [65]:
# ============================================================
# STEP 9 - CREATE FEATURE PREPROCESSING PIPELINE
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            StandardScaler(),
            numerical_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

print("Preprocessing pipeline created successfully.")
print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

Preprocessing pipeline created successfully.
Numerical features: 22
Categorical features: 6


In [66]:
# ============================================================
# STEP 10 - VERIFY PREPROCESSOR CONFIGURATION
# ============================================================

print("Preprocessor configuration:")
print(preprocessor)

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Preprocessor configuration:
ColumnTransformer(transformers=[('numerical', StandardScaler(),
                                 ['Average_Score', 'Average_Attempts',
                                  'Average_Time_Taken', 'Total_Activities',
                                  'Completion_Rate', 'Pre_Test', 'Post_Test',
                                  'Unit_Test', 'Monthly_Test', 'Term_Result',
                                  'Motor_Skills_Score', 'Creativity_Score',
                                  'Interactive_Engagement', 'Test_Improvement',
                                  'Participation_Score', 'Attention_Score',
                                  'Confidence_Score', 'Collaboration_Score',
                                  'Historical_Lessons', 'Average_Sequence_No',
                                  'Completed_Historical_Lessons',
                                  'Historical_Completion_Rate']),
                                ('categorical',
                                 OneH

In [67]:
# ============================================================
# STEP 11 - CREATE MACHINE LEARNING MODEL PIPELINES
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# ------------------------------------------------------------
# 1. Logistic Regression
# ------------------------------------------------------------

logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])


# ------------------------------------------------------------
# 2. Decision Tree
# ------------------------------------------------------------

decision_tree_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        max_depth=4,
        class_weight="balanced",
        random_state=42
    ))
])


# ------------------------------------------------------------
# 3. Random Forest
# ------------------------------------------------------------

random_forest_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=5,
        class_weight="balanced",
        random_state=42
    ))
])


# ------------------------------------------------------------
# Store all models
# ------------------------------------------------------------

models = {
    "Logistic Regression": logistic_pipeline,
    "Decision Tree": decision_tree_pipeline,
    "Random Forest": random_forest_pipeline
}

print("Machine learning pipelines created successfully.")

print("\nModels:")
for name in models:
    print("-", name)

Machine learning pipelines created successfully.

Models:
- Logistic Regression
- Decision Tree
- Random Forest


In [68]:
# ============================================================
# STEP 12 - SET UP LEAVE-ONE-OUT CROSS-VALIDATION
# ============================================================

from sklearn.model_selection import LeaveOneOut

# Create Leave-One-Out Cross-Validation
loo = LeaveOneOut()

print("Cross-validation method: Leave-One-Out Cross-Validation")
print("Number of students:", len(X))
print("Number of validation folds:", loo.get_n_splits(X))

Cross-validation method: Leave-One-Out Cross-Validation
Number of students: 15
Number of validation folds: 15


In [69]:
# ============================================================
# STEP 13 - TRAIN MODELS USING LEAVE-ONE-OUT CROSS-VALIDATION
# ============================================================

from sklearn.model_selection import cross_val_predict

# Store predictions from each model
cv_predictions = {}

for model_name, model in models.items():

    print(f"\nTraining {model_name}...")

    # Generate predictions using Leave-One-Out Cross-Validation
    predictions = cross_val_predict(
        model,
        X,
        y,
        cv=loo
    )

    cv_predictions[model_name] = predictions

    print(f"{model_name} training completed.")


Training Logistic Regression...
Logistic Regression training completed.

Training Decision Tree...
Decision Tree training completed.

Training Random Forest...
Random Forest training completed.


In [70]:
# ============================================================
# STEP 14 - TRAIN FINAL MODELS ON FULL DATASET
# ============================================================

final_models = {}

for model_name, model in models.items():

    print(f"\nTraining final {model_name} model...")

    # Train using the complete dataset
    model.fit(X, y)

    final_models[model_name] = model

    print(f"{model_name} final model trained successfully.")

print("\nAll final models trained successfully.")


Training final Logistic Regression model...
Logistic Regression final model trained successfully.

Training final Decision Tree model...
Decision Tree final model trained successfully.

Training final Random Forest model...
Random Forest final model trained successfully.

All final models trained successfully.


In [71]:
# ============================================================
# STEP 15 - SAVE TRAINED MODELS
# ============================================================

import os
import joblib

# Create models directory if it does not exist
os.makedirs("../models", exist_ok=True)

for model_name, model in final_models.items():

    filename = model_name.lower().replace(" ", "_") + ".joblib"
    filepath = os.path.join("../models", filename)

    joblib.dump(model, filepath)

    print(f"Saved: {filepath}")

print("\nAll trained models saved successfully.")

Saved: ../models/logistic_regression.joblib
Saved: ../models/decision_tree.joblib
Saved: ../models/random_forest.joblib

All trained models saved successfully.


In [72]:
# ============================================================
# STEP 16 - CREATE CROSS-VALIDATION PREDICTION RESULTS
# ============================================================

prediction_results = pd.DataFrame({
    "Student_ID": model_data["Student_ID"],
    "Actual_Outcome": y
})

# Add predictions from each model
for model_name, predictions in cv_predictions.items():
    prediction_results[f"{model_name}_Prediction"] = predictions

print("Cross-validation prediction results:")
display(prediction_results)

Cross-validation prediction results:


,Student_ID,Actual_Outcome,Logistic Regression_Prediction,Decision Tree_Prediction,Random Forest_Prediction
0,ST001,Successful,Successful,Successful,Successful
1,ST002,Successful,Needs Practice,Needs Practice,Needs Practice
2,ST003,Successful,Successful,Successful,Successful
3,ST004,Waiting,Needs Practice,Needs Practice,Needs Practice
4,ST005,Successful,Successful,Successful,Successful
5,ST006,Needs Practice,Needs Practice,Needs Practice,Needs Practice
6,ST007,Needs Practice,Needs Practice,Needs Practice,Needs Practice
7,ST008,Successful,Successful,Successful,Successful
8,ST009,Needs Practice,Successful,Successful,Needs Practice
9,ST010,Successful,Successful,Successful,Successful


In [73]:
# ============================================================
# STEP 17 - CALCULATE CROSS-VALIDATION ACCURACY
# ============================================================

from sklearn.metrics import accuracy_score

model_accuracy = {}

for model_name, predictions in cv_predictions.items():

    accuracy = accuracy_score(y, predictions)

    model_accuracy[model_name] = accuracy

    print(
        f"{model_name}: "
        f"{accuracy:.4f} "
        f"({accuracy * 100:.2f}%)"
    )

Logistic Regression: 0.7333 (73.33%)
Decision Tree: 0.7333 (73.33%)
Random Forest: 0.8000 (80.00%)


In [74]:
# ============================================================
# STEP 18 - COMPARE MODELS AND SELECT BEST MODEL
# ============================================================

import pandas as pd

print("=" * 70)
print("STEP 18 - MODEL PERFORMANCE COMPARISON")
print("=" * 70)

# Create model comparison table
model_comparison = pd.DataFrame(
    list(model_accuracy.items()),
    columns=["Model", "Accuracy"]
)

# Convert accuracy to percentage
model_comparison["Accuracy (%)"] = (
    model_comparison["Accuracy"] * 100
).round(2)

# Rank models from highest to lowest accuracy
model_comparison = model_comparison.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

# Add ranking
model_comparison["Rank"] = range(
    1,
    len(model_comparison) + 1
)

print("\nModel Performance Comparison:")
display(model_comparison)

# Select best model
best_model_name = model_comparison.loc[0, "Model"]

print("\nBest Model:")
print(best_model_name)

print(
    f"Best LOOCV Accuracy: "
    f"{model_comparison.loc[0, 'Accuracy (%)']:.2f}%"
)

STEP 18 - MODEL PERFORMANCE COMPARISON

Model Performance Comparison:


,Model,Accuracy,Accuracy (%),Rank
0,Random Forest,0.800000,80.00,1
1,Logistic Regression,0.733333,73.33,2
2,Decision Tree,0.733333,73.33,3



Best Model:
Random Forest
Best LOOCV Accuracy: 80.00%


In [75]:
# ============================================================
# STEP 19 - SAVE CROSS-VALIDATION RESULTS
# ============================================================

import os

print("=" * 70)
print("STEP 19 - SAVE CROSS-VALIDATION RESULTS")
print("=" * 70)

# Create processed data directory
os.makedirs("../data/processed", exist_ok=True)

# Save cross-validation prediction results
prediction_results.to_csv(
    "../data/processed/cv_prediction_results.csv",
    index=False
)

# Save model comparison results
model_comparison.to_csv(
    "../data/processed/model_comparison.csv",
    index=False
)

print("\n✓ Cross-validation prediction results saved successfully.")
print("✓ Model comparison results saved successfully.")

print("\nSaved files:")
print("1. ../data/processed/cv_prediction_results.csv")
print("2. ../data/processed/model_comparison.csv")

STEP 19 - SAVE CROSS-VALIDATION RESULTS

✓ Cross-validation prediction results saved successfully.
✓ Model comparison results saved successfully.

Saved files:
1. ../data/processed/cv_prediction_results.csv
2. ../data/processed/model_comparison.csv
